<a href="https://colab.research.google.com/github/dhvariya/ChemEng/blob/main/SOEC_Calculations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install cantera
import cantera as ct
import numpy as np

F = 96485.33          # Faraday constant, C/mol
R = 8.314462e-3       # gas constant, kJ/mol/K
gas = ct.Solution('gri30.yaml')        # ideal-gas species data
graphite = ct.Solution('graphite.yaml')  # solid carbon data


def H_G(species, T):
    """Molar enthalpy and Gibbs energy (kJ/mol) of a pure gas at T (K), 1 atm."""
    gas.TPX = T, ct.one_atm, {species: 1.0}
    return gas.enthalpy_mole / 1e6, gas.gibbs_mole / 1e6


def G_graphite(T):
    graphite.TP = T, ct.one_atm
    return graphite.gibbs_mole / 1e6


# ---------------------------------------------------------------
# 1-3. Reaction energies, voltages, equilibrium constants (Fig 2-4, Table 1)
# ---------------------------------------------------------------
print("T(C)   dH_H2O  dG_H2O  Erev_H2O Etn_H2O | dH_CO2  dG_CO2  Erev_CO2 Etn_CO2 | K_RWGS  K_Boud")
for T in range(400, 1301, 50):
    hH2O, gH2O = H_G('H2O', T); hH2, gH2 = H_G('H2', T); hO2, gO2 = H_G('O2', T)
    hCO2, gCO2 = H_G('CO2', T); hCO, gCO = H_G('CO', T)

    # H2O -> H2 + 1/2 O2
    dH1 = hH2 + 0.5 * hO2 - hH2O
    dG1 = gH2 + 0.5 * gO2 - gH2O
    # CO2 -> CO + 1/2 O2
    dH2 = hCO + 0.5 * hO2 - hCO2
    dG2 = gCO + 0.5 * gO2 - gCO2

    # Voltages: E = energy / (n F), n = 2 electrons
    Erev1, Etn1 = dG1 * 1000 / (2 * F), dH1 * 1000 / (2 * F)
    Erev2, Etn2 = dG2 * 1000 / (2 * F), dH2 * 1000 / (2 * F)

    # RWGS: CO2 + H2 = CO + H2O ;  Boudouard: 2CO = CO2 + C(s)
    dG_rwgs = (gCO + gH2O) - (gCO2 + gH2)
    dG_boud = (gCO2 + G_graphite(T)) - 2 * gCO
    K_rwgs = np.exp(-dG_rwgs / (R * T))
    K_boud = np.exp(-dG_boud / (R * T))

    print(f"{T-273.15:6.1f} {dH1:7.1f} {dG1:7.1f} {Erev1:8.3f} {Etn1:7.3f} | "
          f"{dH2:7.1f} {dG2:7.1f} {Erev2:8.3f} {Etn2:7.3f} | {K_rwgs:7.3f} {K_boud:7.3g}")


# ---------------------------------------------------------------
# 4-5. Outlet composition and carbon activity (Fig 5-6)
# ---------------------------------------------------------------
def outlet(ratio, conversion, T_C, x_H2_feed=0.10):
    """
    Feed: 1 mol total = 10% H2 + (H2O + CO2) at H2O:CO2 = ratio.
    The electrolyser removes oxygen equal to `conversion` x mol(H2O + CO2).
    The remaining H, C, O atoms are equilibrated (Gibbs minimisation) at T_C, 1 atm.
    """
    n_re = 1 - x_H2_feed
    nH2O = n_re * ratio / (1 + ratio)
    nCO2 = n_re / (1 + ratio)
    H = 2 * nH2O + 2 * x_H2_feed
    C = nCO2
    O = nH2O + 2 * nCO2 - conversion * (nH2O + nCO2)   # oxygen left after O2- removal

    # Any starting mixture with the right H, C, O atoms works; equilibrium finds the rest
    start = {'CO': C, 'H2O': O - C, 'H2': H / 2 - (O - C)}
    T = T_C + 273.15
    gas.TPX = T, ct.one_atm, start
    gas.equilibrate('TP')
    x = {s: gas[s].X[0] for s in ['H2', 'CO', 'H2O', 'CO2', 'CH4']}

    # Carbon activity from Boudouard equilibrium: a_C = K_B * pCO^2 / pCO2
    _, gCO = H_G('CO', T); _, gCO2 = H_G('CO2', T)
    K_B = np.exp(-((gCO2 + G_graphite(T)) - 2 * gCO) / (R * T))
    a_C = K_B * x['CO'] ** 2 / x['CO2']
    return x, a_C


print("\nFig 5: outlet H2/CO (70% conversion)")
for T_C in (750, 800, 850):
    for r in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
        x, _ = outlet(r, 0.70, T_C)
        print(f"  {T_C} C  H2O:CO2 = {r:3.1f}  ->  H2/CO = {x['H2']/x['CO']:.2f}  (CH4 = {x['CH4']*100:.2f} mol%)")

print("\nFig 6: carbon activity (feed H2O:CO2 = 2)")
for T_C in (650, 700, 750, 800):
    for U in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.93, 0.95, 0.97, 0.98, 0.99, 0.995]:
        _, aC = outlet(2.0, U, T_C)
        print(f"  {T_C} C  conversion {U*100:5.1f}%  a_C = {aC:.3f}{'  <-- carbon possible' if aC > 1 else ''}")


# ---------------------------------------------------------------
# 6. Electricity per Nm3 syngas, H2/CO = 2 (Fig 7, Table 2)
# ---------------------------------------------------------------
T = 1073.15
dH_H2O = H_G('H2', T)[0] + 0.5 * H_G('O2', T)[0] - H_G('H2O', T)[0]
dH_CO2 = H_G('CO', T)[0] + 0.5 * H_G('O2', T)[0] - H_G('CO2', T)[0]
V_soec = (2 * dH_H2O + dH_CO2) / 3 * 1000 / (2 * F)        # mixed thermoneutral voltage
dH_rwgs_900 = (H_G('CO', 1173.15)[0] + H_G('H2O', 1173.15)[0]
               - H_G('CO2', 1173.15)[0] - H_G('H2', 1173.15)[0])

eta = 0.95        # rectifier efficiency (assumption)
V_pem = 1.80      # PEM cell voltage [19]
V_lt, FE = 3.6, 0.90   # low-temperature CO2 cell [27]
L_vap = 44.0      # latent heat of water, kJ/mol
nF = 2 * F / 1000      # kJ per volt per mol of H2 or CO

case = {}
case['1 SOEC, waste-heat steam'] = nF * V_soec / eta
case['2 SOEC, electric steam'] = case['1 SOEC, waste-heat steam'] + (2 / 3) * L_vap
case['3 PEM + RWGS'] = nF * V_pem / eta + (1 / 3) * dH_rwgs_900     # 1 mol H2 per mol syngas
byproduct_H2 = (1 / 3) * (1 - FE) / FE
case['4 LT CO2 + PEM'] = ((1 / 3) * nF * V_lt / FE + (2 / 3 - byproduct_H2) * nF * V_pem) / eta

to_kWh_per_Nm3 = 44.615 / 3600          # kJ/mol -> kWh/Nm3
LHV = (2 * 241.83 + 282.98) / 3         # kJ/mol syngas
print(f"\nMixed thermoneutral voltage at 800 C = {V_soec:.3f} V")
for k, v in case.items():
    print(f"  {k:26s} {v:6.1f} kJ/mol  {v*to_kWh_per_Nm3:5.2f} kWh/Nm3  LHV eff {LHV/v*100:5.1f}%")


# ---------------------------------------------------------------
# 7. Break-even grid intensity vs fossil syngas (Fig 8, Table 3)
# ---------------------------------------------------------------
kgH2_per_Nm3 = 44.615 * 2.016 / 1000      # 1 mol syngas = 1 mol H2-equivalent
fossil = {'SMR': (10, 12), 'Coal': (22, 26)}   # kg CO2-eq / kg H2, IEA [28]
print("\nBreak-even electricity intensity (g CO2-eq/kWh)")
for k, v in case.items():
    kwh = v * to_kWh_per_Nm3
    s = "  ".join(f"{f}: {lo*kgH2_per_Nm3/kwh*1000:.0f}-{hi*kgH2_per_Nm3/kwh*1000:.0f}"
                  for f, (lo, hi) in fossil.items())
    print(f"  {k:26s} {s}")


T(C)   dH_H2O  dG_H2O  Erev_H2O Etn_H2O | dH_CO2  dG_CO2  Erev_CO2 Etn_CO2 | K_RWGS  K_Boud
 126.9   242.9   223.9    1.160   1.258 |   283.5   248.3    1.287   1.469 |   0.001 2.07e+13
 176.9   243.4   221.5    1.148   1.261 |   283.6   243.9    1.264   1.470 |   0.003 6.29e+10
 226.9   243.8   219.0    1.135   1.264 |   283.7   239.5    1.241   1.470 |   0.007 6.07e+08
 276.9   244.3   216.5    1.122   1.266 |   283.7   235.1    1.218   1.470 |   0.017 1.36e+07
 326.9   244.8   214.0    1.109   1.268 |   283.6   230.7    1.195   1.470 |   0.035 5.76e+05
 376.9   245.2   211.4    1.096   1.271 |   283.6   226.3    1.172   1.470 |   0.064 3.98e+04
 426.9   245.6   208.8    1.082   1.273 |   283.5   221.8    1.150   1.469 |   0.106 4.03e+03
 476.9   246.0   206.2    1.068   1.275 |   283.4   217.4    1.127   1.469 |   0.163     557
 526.9   246.4   203.5    1.054   1.277 |   283.3   213.1    1.104   1.468 |   0.237    98.7
 576.9   246.8   200.8    1.040   1.279 |   283.1   208.7    1.0